# GraviFrame - DataFrames and time series

*Dibuat oleh Gravicode Studios, dipimpin oleh Kang Fadhil*

In [ ]:
#r "../src/GraviNum/bin/Release/net10.0/Gravicode.Science.GraviNum.dll"
#r "../src/GraviFrame/bin/Release/net10.0/Gravicode.Science.GraviFrame.dll"
#r "nuget: ScottPlot, 5.1.59"

using Gravicode.Science.GraviFrame;
using Gravicode.Science.GraviNum;

var titanic = DataFrame.ReadCsv("../datasets/titanic.csv");
Console.WriteLine(titanic.SelectColumns("survived", "pclass", "sex", "age", "fare").ToString(10));

## Missing values

In [ ]:
foreach (var (column, missing) in titanic.MissingCounts().Where(kv => kv.Value > 0))
    Console.WriteLine($"{column,-16}{missing,5} missing");

var clean = titanic.WithColumn(titanic.Numeric("age").FillMissingWithMedian().Rename("age_filled"));

## GroupBy and pivot

In [ ]:
Console.WriteLine(clean.GroupBy("pclass", "sex").Mean("survived").ToString(10));
Console.WriteLine();
Console.WriteLine(clean.Pivot("pclass", "sex", "survived").ToString());

## Time series

Rolling windows, percentage change and calendar resampling.

In [ ]:
var prices = DataFrame.ReadCsv("../datasets/finance_timeseries.csv");
var grvc = prices.Filter(r => r.String("ticker") == "GRVC").SortBy("date");
var close = grvc.Numeric("close");

var enriched = grvc
    .WithColumn(close.Rolling(7).Mean().Rename("ma7"))
    .WithColumn(close.Rolling(30).Mean().Rename("ma30"))
    .WithColumn(close.PercentChange().Rename("daily_return"));

Console.WriteLine(enriched.SelectColumns("date", "close", "ma7", "ma30").Tail(6).ToString());
Console.WriteLine($"annualised volatility: {enriched.Numeric("daily_return").Std() * Math.Sqrt(252):P2}");

## Trend chart

In [ ]:
var days = Enumerable.Range(0, grvc.RowCount).Select(i => (double)i).ToArray();

var plot = new ScottPlot.Plot();
var c = plot.Add.Scatter(days, close.Values); c.LegendText = "close"; c.MarkerSize = 0;
var m7 = plot.Add.Scatter(days, enriched.Numeric("ma7").Values); m7.LegendText = "7-day"; m7.MarkerSize = 0;
var m30 = plot.Add.Scatter(days, enriched.Numeric("ma30").Values); m30.LegendText = "30-day"; m30.MarkerSize = 0;

plot.Title("GRVC close with rolling averages");
plot.ShowLegend();
plot.GetImageHtml(950, 500)

## Describe and correlate

In [ ]:
Console.WriteLine(clean.SelectColumns("survived", "pclass", "age_filled", "fare").Describe().ToString());
Console.WriteLine();
Console.WriteLine(clean.SelectColumns("survived", "pclass", "age_filled", "fare").CorrelationMatrix().ToString());